# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

---

**Note:** All environment and hyperparameter settings are now collected in a single `CONFIG` dictionary at the top of the notebook. To change the environment or any hyperparameter, simply edit the values in the config cell.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [ ]:
%pip install gymnasium stable-baselines3 wandb tsilva-notebook-utils==0.0.121 --quiet

In [ ]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY'
])

In [ ]:
from tsilva_notebook_utils.torch import get_default_device
DEVICE = get_default_device()
DEVICE

## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [ ]:
import torch.nn as nn
from tsilva_notebook_utils.gymnasium import build_env as _build_env, set_random_seed

# --- Config dictionary for all hyperparameters and environment settings ---
def setup_config(env_id):
    common = dict(
        env_id=env_id,         # Environment name
        max_epochs=-1,          # Maximum number of training epochs (-1 for no limit)
        seed=42,               # Random seed for reproducibility
        gamma=0.99,            # Discount factor for future rewards
        lam=0.95,              # GAE lambda for advantage estimation
        clip_epsilon=0.2,      # PPO clip range for policy update
        minibatch_size=64,     # Minibatch size for SGD
        train_rollout_steps=2048,  # Number of steps to collect per training rollout
        eval_interval=10,       # Evaluate every N epochs
        eval_episodes=32,      # Number of episodes for evaluation
        reward_threshold=200,  # Reward threshold to consider environment solved
        policy_lr=3e-4,        # Learning rate for policy network
        value_lr=1e-3,         # Learning rate for value network
        hidden_dim=64,         # Hidden layer size(s) for networks - can be int or tuple (e.g., (64, 32) for two layers)
        entropy_coef=0.01,     # Coefficient for entropy bonus (encourages exploration)
        normalize=False,       # Whether to use input normalization
        mean_reward_window=100,  # Window size for mean reward calculation (for early stopping)
        rollout_interval=10,   # Epoch interval for collecting rollouts
        n_envs="auto",         # Maximum number of parallel environments
        async_rollouts=True    # Use async rollout collection (True) or sync (False)
    )
    env_specific = {
        # In your CONFIG for CartPole-v1:
        "CartPole-v1": dict(
            # Reduce rollout steps - CartPole episodes are short (~200 steps max)
            train_rollout_steps=512,        # Down from 64 (was too small)
            
            # Increase minibatch size for better GPU utilization
            minibatch_size=256,             # Up from 1024 (better for smaller rollouts)
            
            # More frequent rollout collection
            rollout_interval=1,             # Down from 8 (collect fresh data more often)
            
            # Reduce evaluation frequency 
            eval_interval=20,               # Up from 10 (less frequent eval)
            eval_episodes=5,                # Down from 10 (faster eval)
            
            # Tighter early stopping
            mean_reward_window=50,          # Down from 100 (faster convergence detection)
            reward_threshold=475,           # Standard CartPole threshold
            
            # Optimized learning rates for faster convergence
            policy_lr=1e-3,                 # Up from 3e-4 (faster learning)
            value_lr=1e-3,                  # Up from 3e-4 (matched to policy)
            
            # Reduce network complexity - can use tuple for multiple layers
            hidden_dim=32,                  # Down from 64 (CartPole is simple) - can also use (64, 32) for two layers
        ),
        "LunarLander-v3": dict(
            gamma=0.99,           # Standard discount for LunarLander
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for LunarLander
            minibatch_size=64,    # Larger batch for more stable updates
            eval_interval=2,      # Evaluate every 2 epochs
            reward_threshold=200, # Solved threshold for LunarLander-v3
            policy_lr=1e-4,       # Lower LR for more complex env
            value_lr=5e-4,        # Lower LR for value net
            hidden_dim=32, # Larger net with two layers for more complex env
            entropy_coef=0.02     # Higher entropy for more exploration
        ),
        "Acrobot-v1": dict(
            gamma=0.99,           # Standard discount for Acrobot
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Acrobot
            minibatch_size=32,    # Smaller batch for faster updates
            eval_interval=2,      # Evaluate more frequently for fast convergence
            reward_threshold=-100, # Solved threshold for Acrobot-v1 (average reward > -100)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for Acrobot
            entropy_coef=0.01     # Typical entropy for Acrobot
        ),
        "Pendulum-v1": dict(
            gamma=0.99,           # Standard discount for Pendulum
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Pendulum
            minibatch_size=64,    # Larger batch for continuous action
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=-200, # Solved threshold for Pendulum-v1 (average reward > -200)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=(128, 64), # Larger net with two layers for continuous control
            entropy_coef=0.0      # No entropy for deterministic continuous control
        ),
        "MountainCar-v0": dict(
            gamma=0.99,             # Discount factor (keep)
            lam=0.97,               # Slightly higher GAE lambda for more bias reduction
            clip_epsilon=0.15,      # Tighter PPO clip for more stable updates
            minibatch_size=16,      # Smaller minibatch for more frequent updates
            eval_interval=2,        # Keep frequent evaluation
            eval_episodes=10,       # Keep
            reward_threshold=-110,  # Keep
            policy_lr=1e-4,         # Lower learning rate for more stable policy updates
            value_lr=5e-4,          # Lower value net LR for stability
            hidden_dim=(128, 64),   # Larger network with two layers for more capacity
            entropy_coef=0.05       # Higher entropy for hard exploration
        ),
    }

    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")
    config = {**common, **env_specific[env_id]}

    return config

ENV_ID = "CartPole-v1"
#ENV_ID = "Acrobot-v1"
#ENV_ID = "LunarLander-v3"
#ENV_ID = "Pendulum-v1"
#ENV_ID = "MountainCar-v0"
CONFIG = setup_config(ENV_ID)
CONFIG

In [ ]:
from tsilva_notebook_utils.gymnasium import log_env_info

# Set random seed for reproducibility
set_random_seed(CONFIG['seed'])

# Wrap build env with config parameters
build_env = lambda seed, n_envs=None: _build_env(
    CONFIG['env_id'], 
    norm_obs=CONFIG['normalize'], 
    n_envs=n_envs if n_envs is not None else CONFIG['n_envs'], 
    seed=seed
)

# Test building env
env = build_env(CONFIG['seed'])
log_env_info(env)

## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [ ]:
class MLPNet(nn.Module):
    """Reusable MLP with configurable hidden dimensions"""
    
    def __init__(self, input_dim, output_dim, hidden_dim=64, activation=nn.ReLU):
        super().__init__()
        
        if isinstance(hidden_dim, (int, float)):
            hidden_dims = [int(hidden_dim)]
        else:
            hidden_dims = [int(dim) for dim in hidden_dim]
        
        layers = []
        current_dim = input_dim
        
        for hidden_size in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_size),
                activation()
            ])
            current_dim = hidden_size
        
        layers.append(nn.Linear(current_dim, output_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

class PolicyNet(MLPNet):
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super().__init__(obs_dim, act_dim, hidden_dim)

class ValueNet(MLPNet):
    def __init__(self, obs_dim, hidden_dim=64):
        super().__init__(obs_dim, 1, hidden_dim)

import torch
from torch.distributions import Categorical

class PPOLoss:
    def __init__(self, clip_epsilon, entropy_coef):
        self.clip_epsilon = clip_epsilon
        self.entropy_coef = entropy_coef
    
    def compute(self, states, actions, old_logps, advantages, returns, policy_model, value_model):
        # Policy loss
        logits = policy_model(states)
        dist = Categorical(logits=logits)
        new_logps = dist.log_prob(actions)
        
        ratio = torch.exp(new_logps - old_logps)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon) * advantages
        entropy = dist.entropy().mean()
        
        policy_loss = -torch.min(surr1, surr2).mean() - self.entropy_coef * entropy
        
        # Value loss
        value_pred = value_model(states).squeeze()
        value_loss = 0.5 * ((returns - value_pred) ** 2).mean()
        
        # Metrics
        clip_fraction = ((ratio < 1.0 - self.clip_epsilon) | (ratio > 1.0 + self.clip_epsilon)).float().mean()
        kl_div = (old_logps - new_logps).mean()
        approx_kl = ((ratio - 1) - torch.log(ratio)).mean()
        explained_var = 1 - torch.var(returns - value_pred) / torch.var(returns)
        
        return {
            'policy_loss': policy_loss,
            'value_loss': value_loss,
            'entropy': entropy,
            'clip_fraction': clip_fraction,
            'kl_div': kl_div,
            'approx_kl': approx_kl,
            'explained_var': explained_var
        }

In [ ]:
import time
import torch
import multiprocessing
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from collections import deque
import threading
import queue
import copy
from tsilva_notebook_utils.gymnasium import RolloutDataset, collect_rollouts, group_trajectories_by_episode

class BaseRolloutCollector:
    """Base class for rollout collectors"""
    def __init__(self, build_env_fn, config, obs_dim, act_dim):
        self.build_env_fn = build_env_fn
        self.config = config
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.last_obs = None
        
    def start(self):
        """Start the collector"""
        pass
        
    def stop(self):
        """Stop the collector"""
        pass
        
    def update_models(self, policy_state_dict, value_state_dict):
        """Update model weights"""
        pass
        
    def get_rollout(self, timeout=1.0):
        """Get next rollout data"""
        raise NotImplementedError
        
    def initialize_with_models(self, policy_model, value_model):
        """Initialize collector with model references"""
        pass

class SyncRolloutCollector(BaseRolloutCollector):
    """Synchronous rollout collector - collects data on demand"""
    def __init__(self, build_env_fn, config, obs_dim, act_dim):
        super().__init__(build_env_fn, config, obs_dim, act_dim)
        self.env = build_env_fn(config['seed'])
        self.policy_model = None
        self.value_model = None
        self._ready_for_initial = False
        
    def initialize_with_models(self, policy_model, value_model):
        """Set model references for sync collector"""
        self.policy_model = policy_model
        self.value_model = value_model
        self._ready_for_initial = True
        
    def get_rollout(self, timeout=1.0):
        """Collect rollout synchronously using current models"""
        if self.policy_model is None or self.value_model is None:
            return None
            
        trajectories, extras = collect_rollouts(
            self.env,
            self.policy_model,
            self.value_model,
            n_steps=self.config['train_rollout_steps'],
            last_obs=self.last_obs
        )
        
        self.last_obs = extras['last_obs']
        return trajectories
        
    def is_ready_for_initial_rollout(self):
        """Check if ready for initial rollout collection"""
        return self._ready_for_initial

class AsyncRolloutCollector(BaseRolloutCollector):
    """Background thread that continuously collects rollouts using latest model weights"""
    def __init__(self, build_env_fn, config, obs_dim, act_dim):
        super().__init__(build_env_fn, config, obs_dim, act_dim)
        
        # Thread-safe queue for rollout data
        self.rollout_queue = queue.Queue(maxsize=3)  # Buffer 3 rollouts max
        
        # Shared model weights (CPU copies for thread safety)
        self.policy_state_dict = None
        self.value_state_dict = None
        self.model_lock = threading.Lock()
        
        # Control flags
        self.running = False
        self.thread = None
        
        # Create environment and models for rollout collection
        self.env = None
        self.policy_model = None
        self.value_model = None
        
    def initialize_with_models(self, policy_model, value_model):
        """Initialize with model state dicts for async collector"""
        self.update_models(policy_model.state_dict(), value_model.state_dict())
        
    def start(self):
        """Start the background rollout collection thread"""
        if self.running:
            return
            
        self.running = True
        self.thread = threading.Thread(target=self._collect_loop, daemon=True)
        self.thread.start()
        
    def stop(self):
        """Stop the background rollout collection"""
        self.running = False
        if self.thread:
            self.thread.join(timeout=5.0)
            
    def update_models(self, policy_state_dict, value_state_dict):
        """Update model weights from main training thread"""
        with self.model_lock:
            self.policy_state_dict = copy.deepcopy(policy_state_dict)
            self.value_state_dict = copy.deepcopy(value_state_dict)
            
    def get_rollout(self, timeout=1.0):
        """Get next rollout data (non-blocking with timeout)"""
        try:
            return self.rollout_queue.get(timeout=timeout)
        except queue.Empty:
            return None
            
    def is_ready_for_initial_rollout(self):
        """Check if ready for initial rollout collection"""
        return self.policy_state_dict is not None and self.value_state_dict is not None

    # TODO: called how many times?
    def _init_models(self):
        """Initialize models in the worker thread"""
        if self.env is None:
            self.env = self.build_env_fn(self.config['seed'] + 1000)  # Different seed for rollout env
        
        # TODO: create from outside?
        if self.policy_model is None:
            self.policy_model = PolicyNet(self.obs_dim, self.act_dim, self.config['hidden_dim'])
            self.policy_model.eval()  # Always in eval mode for rollouts
            
        # TODO: create from outside?
        if self.value_model is None:
            self.value_model = ValueNet(self.obs_dim, self.config['hidden_dim'])
            self.value_model.eval()
            
    def _update_model_weights(self):
        """Update local model weights from shared state dicts"""
        with self.model_lock:
            if self.policy_state_dict is not None:
                self.policy_model.load_state_dict(self.policy_state_dict)
            if self.value_state_dict is not None:
                self.value_model.load_state_dict(self.value_state_dict)
                
    def _collect_loop(self):
        """Main loop running in background thread"""
        self._init_models()
        
        while self.running:
            try:
                # Update to latest model weights
                self._update_model_weights()
                
                # Collect rollout
                trajectories, extras = collect_rollouts(
                    self.env,
                    self.policy_model,
                    self.value_model,
                    n_steps=self.config['train_rollout_steps'],
                    last_obs=self.last_obs
                )
                
                self.last_obs = extras['last_obs']
                
                # Put rollout in queue (non-blocking, drop if full)
                try:
                    self.rollout_queue.put(trajectories, block=False)
                except queue.Full:
                    # Queue is full, drop oldest and add new
                    try:
                        self.rollout_queue.get_nowait()
                        self.rollout_queue.put(trajectories, block=False)
                    except queue.Empty:
                        pass
                        
            except Exception as e:
                print(f"Error in rollout collection: {e}")
                time.sleep(0.1)  # Brief pause on error

# ---------------------------------------------------------------------
#  PPO Lightning module with fully unified rollout collection
# ---------------------------------------------------------------------
class PPOAgent(pl.LightningModule):
    def __init__(self, obs_dim, act_dim, config):
        super().__init__()
        self.save_hyperparameters()

        # ----------------- unpack config -----------------
        self.config = config
        self.entropy_coef   = config['entropy_coef']
        self.clip_epsilon   = config['clip_epsilon']
        self.gamma          = config['gamma']
        self.lam            = config['lam']
        self.minibatch_size     = config['minibatch_size']
        self.eval_interval      = config['eval_interval']
        self.eval_episodes      = config['eval_episodes']
        self.reward_threshold   = config['reward_threshold']
        self.policy_lr          = config['policy_lr']
        self.value_lr           = config['value_lr']
        self.mean_reward_window = config['mean_reward_window']
        self.rollout_interval      = config['rollout_interval']
        self.train_rollout_steps = config['train_rollout_steps']
        self.async_rollouts = config['async_rollouts']

        # ----------------- models & env -----------------
        self.policy_model = PolicyNet(
            obs_dim, act_dim,
            hidden_dim=config['hidden_dim']
        )
        self.value_model = ValueNet(
            obs_dim,
            hidden_dim=config['hidden_dim']
        )
        self.env = build_env(config['seed'])
        self.obs_dim, self.act_dim = obs_dim, act_dim

        # ----------------- PPO loss function -----------------
        self.ppo_loss = PPOLoss(self.clip_epsilon, self.entropy_coef)

        # ----------------- rollout storage --------------
        self.rollout_ds = RolloutDataset()
        self.episode_reward_deque = deque(maxlen=self.mean_reward_window)
        
        # Initialize appropriate rollout collector based on mode
        rollout_collector_cls = AsyncRolloutCollector if self.async_rollouts else SyncRolloutCollector
        self.rollout_collector = rollout_collector_cls(
            build_env, config, obs_dim, act_dim
        )
        
        # Disable automatic optimization to allow maintaining a 
        # different optimizer for each model (policy and value)
        self.automatic_optimization = False
        self.training_start_time = None
        self.training_end_time = None

    # ===================================================
    #  Lightning hooks
    # ===================================================
    def setup(self, stage: str):
        """Initialize rollout collection"""
        if stage == "fit":
            # Initialize collector with models
            self.rollout_collector.initialize_with_models(self.policy_model, self.value_model)
            
            # Start collector
            self.rollout_collector.start()
            
            # Wait for collector to be ready and collect initial rollout
            print("Waiting for initial rollout...")
            while True:
                if self.rollout_collector.is_ready_for_initial_rollout():
                    trajectories = self.rollout_collector.get_rollout(timeout=2.0)
                    if trajectories is not None:
                        self.rollout_ds.update(*trajectories)
                        episodes = group_trajectories_by_episode(trajectories)
                        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
                        for r in episode_rewards:  self.episode_reward_deque.append(float(r))
                        break
                print("Still waiting for rollout...")

    def train_dataloader(self):
        """Standard DataLoader built *once*; dataset is mutable."""
        return DataLoader(
            self.rollout_ds,
            batch_size=self.minibatch_size,
            shuffle=True,
            pin_memory=True if self.device.type != 'mps' else False,
            num_workers=multiprocessing.cpu_count() // 2 if self.device.type != 'mps' else 0
        )

    def on_fit_start(self):
        """Called when training starts"""
        self.training_start_time = time.time()
        mode = "async" if self.async_rollouts else "sync"
        print(f"PPO training started in {mode} mode at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    def on_fit_end(self):
        """Called when training ends"""
        self.rollout_collector.stop()
        self.training_end_time = time.time()
        total_time = self.training_end_time - self.training_start_time
        print(f"PPO training completed in {total_time:.2f} seconds ({total_time/60:.2f} minutes)")

    def on_train_epoch_start(self):
        # Update models with latest weights
        self.rollout_collector.update_models(
            self.policy_model.state_dict(),
            self.value_model.state_dict()
        )
        
        # In case no collection should happen this epoch then skip
        should_collect = (self.current_epoch + 1) % self.rollout_interval == 0    
        if not should_collect: return

        timeout = 2.0 if self.async_rollouts else 1.0  # Longer timeout for async when we actually want data
        trajectories = self.rollout_collector.get_rollout(timeout=timeout)
        if trajectories is not None:
            self.rollout_ds.update(*trajectories)
            episodes = group_trajectories_by_episode(trajectories)
            episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
            for r in episode_rewards:  self.episode_reward_deque.append(float(r))
            self.log('rollout/queue_updated', 1.0)
        else:
            self.log('rollout/queue_miss', 1.0)

    def on_train_epoch_end(self):
        """Evaluate model every N epochs and check for early stopping."""
        should_eval = (self.current_epoch + 1) % self.eval_interval == 0
        if should_eval: self._evaluate_model()

    # ---------------------------------------------------
    def training_step(self, batch, batch_idx):
        opt_policy, opt_value = self.optimizers()

        # unpack batch
        (states, actions, rewards, dones, old_logps, values, advantages, returns, frames) = batch

        # Compute losses and metrics using PPOLoss
        loss_results = self.ppo_loss.compute(
            states, actions, old_logps, advantages, returns, 
            self.policy_model, self.value_model
        )

        policy_loss = loss_results['policy_loss']
        value_loss = loss_results['value_loss']

        # ---------- optimizers ----------
        opt_policy.zero_grad()
        self.manual_backward(policy_loss)
        opt_policy.step()

        opt_value.zero_grad()
        self.manual_backward(value_loss)
        opt_value.step()

        # Additional useful metrics
        advantage_mean = advantages.mean()
        advantage_std = advantages.std()
        value_mean = values.mean()
        returns_mean = returns.mean()
        
        mean_reward = np.mean(self.episode_reward_deque) if len(self.episode_reward_deque) >= self.mean_reward_window else 0

        # Enhanced logging
        self._log_dict({
            'train/mean_reward': mean_reward,
            'train/policy_loss': policy_loss,
            'train/value_loss': value_loss,
            'train/entropy': loss_results['entropy'],
            'train/kl_divergence': loss_results['kl_div'],
            'train/explained_variance': loss_results['explained_var']
        }, prog_bar=True)
        
        self._log_dict({
            'train/approx_kl': loss_results['approx_kl'],
            'train/clip_fraction': loss_results['clip_fraction'],
            'train/advantage_mean': advantage_mean,
            'train/advantage_std': advantage_std,
            'train/value_mean': value_mean,
            'train/returns_mean': returns_mean,
        }, prog_bar=False)

        return policy_loss + value_loss

    def configure_optimizers(self):
        return [
            torch.optim.Adam(self.policy_model.parameters(), lr=self.policy_lr),
            torch.optim.Adam(self.value_model.parameters(), lr=self.value_lr)
        ]

    def forward(self, x):
        return self.policy_model(x)

    def _log_dict(self, dict, **kwargs):
        _dict = {k: v for k, v in dict.items() if v is not None}
        self.log_dict(_dict, **kwargs)

    def _evaluate_model(self):
        eval_seed = np.random.randint(0, 1_000_000)
        eval_env = build_env(eval_seed)
        self.policy_model.eval()
        try: 
            return self.__evaluate_model(eval_env)
        finally: 
            self.policy_model.train()
            eval_env.close()

    def __evaluate_model(self, env):
        # Collect rollouts and add them to 
        # dataset for sampling during training step
        start = time.time()
        trajectories, _ = collect_rollouts(
            env,
            self.policy_model,
            self.value_model,
            n_episodes=self.eval_episodes, 
            deterministic=False,
        )
        end = time.time()
        elapsed = end - start

        # Add mean episode rewards to deque 
        # (to be able to average over N episodes)
        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        mean_episode_reward = np.mean(episode_rewards)

        num_steps = len(trajectories[0])
        num_episodes = len(episodes)

        self._log_dict({
            'rollout/mean_reward': mean_episode_reward,
            'rollout/num_episodes': num_episodes,
            'rollout/num_steps': num_steps,
            'rollout/avg_steps_per_episode': num_steps / (num_episodes + 1e-3),
            'rollout/time_elapsed': elapsed,
            'rollout/steps_per_second': num_steps / (elapsed + 1e-3)
        })

        episodes = group_trajectories_by_episode(trajectories)
        eval_mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
        self.log('eval/mean_reward', eval_mean_reward, prog_bar=True)
            
        # Check for early stopping based on eval reward
        if eval_mean_reward >= self.reward_threshold:
            print(f"Early stopping at epoch {self.current_epoch} with eval mean reward {eval_mean_reward:.2f} >= threshold {self.reward_threshold}")
            self.trainer.should_stop = True

## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger

# Create PPO agent and move to device
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n if hasattr(env.action_space, 'n') else env.action_space.shape[0]  # Handle discrete and continuous actions
ppo_agent = PPOAgent(obs_dim, act_dim, CONFIG)

# Set up trainer with proper device configuration
wandb_logger = WandbLogger(project="gymnasium_ppo") # TODO: softcode this

trainer = Trainer(
    logger=wandb_logger,
    log_every_n_steps=10,
    max_epochs=CONFIG['max_epochs'],
    enable_progress_bar=True,
    enable_checkpointing=False,  # Disable checkpointing for speed
    accelerator="auto"
)

# Fit the model
trainer.fit(ppo_agent)

In [ ]:
import random
from tsilva_notebook_utils.gymnasium import render_episode_frames

n_episodes = 8
trajectories, _ = collect_rollouts(
    build_env(random.randint(0, 1_000_000), n_envs=n_episodes),
    ppo_agent.policy_model,
    n_episodes=n_episodes,
    deterministic=True,
    collect_frames=True
)
episodes = group_trajectories_by_episode(trajectories) # something is wrong in frame collection
mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
episode_frames = [[step[-1] for step in episode] for episode in episodes]
print(f"Mean reward: {mean_reward:.2f}")
render_episode_frames(episode_frames, out_dir="./tmp", grid=(2, 2), text_color=(0, 0, 0))

TODO: train for convergence without deterministic policy in 
TODO: stream envs, fix training bottleneck
TODO: Early stopping at epoch 39 with eval mean reward 721.80 (wrong)